# Lab 1A: Ingest CSV files into Bronze

Attach `seer_livelabs_spark`. Run the cells in order to load five CSV files into five Bronze Delta tables. This notebook only loads source data; it has no validation task or Silver/Gold processing. Each load recreates only its named workshop table without a backup. Source CSV files are not changed.

After this notebook, run **Lab 1B: Transform Bronze into Silver**.

## Rerunning Lab 1A

No manual deletion is required. Each load cell drops and recreates its named Bronze table without a backup; the five source CSVs, schemas, volumes and unrelated objects remain unchanged. A failed write after a drop may leave that table absent.

Start at setup and run the load cells in order. After changing sources or rebuilding Bronze, continue **Lab 1B → Lab 1C → Lab 2** so downstream data and predictions match. Existing downstream tables do not refresh automatically. Resolve any failed write before continuing. Run one writer at a time, then refresh the catalog and verify counts and lineage.

Lab 2 reuses its experiment, adds new training runs/model versions and overwrites predictions; do not delete experiment history as part of a Lab 1 reset. See the **Reruns and recovery** section of the repository's root `README.md` for both labs' rerun and cleanup guidance.


## Task 1: Load the source files

Set the input and destination locations. Then run each CSV load below. All source columns stay as strings; types and business rules are applied in Silver.


In [ ]:
import json
from pathlib import Path

# Use the shared settings for the catalog and source-volume location.
config = json.loads(Path("/Workspace/Shared/seer-aidp-livelabs/config/workshop.json").read_text())
BRONZE = f"{config['catalog']}.{config['schemas']['bronze']}"
SOURCE_ROOT = config["source_root"]

# Create the bronze schema if it does not already exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")


### suppliers_raw


In [ ]:
# Read the CSV as strings; keep the supplied values without business transformations.
source_file = "source-data/suppliers/supplier_extract.csv"
if config["source_layout"] == "flat":
    source_file = source_file.replace("/", "_")
raw = spark.read.csv(f"{SOURCE_ROOT}/{source_file}", header=True, inferSchema=False, mode="FAILFAST").fillna("")
# Recreate only this Bronze table so reruns start with a fresh table identity.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.suppliers_raw")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.suppliers_raw")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {BRONZE}.suppliers_raw (up to 5 records):")
spark.table(f"{BRONZE}.suppliers_raw").show(5, truncate=False)


### assets_raw


In [ ]:
# Read the CSV as strings; keep the supplied values without business transformations.
source_file = "source-data/assets/financial_assets.csv"
if config["source_layout"] == "flat":
    source_file = source_file.replace("/", "_")
raw = spark.read.csv(f"{SOURCE_ROOT}/{source_file}", header=True, inferSchema=False, mode="FAILFAST").fillna("")
# Recreate only this Bronze table so reruns start with a fresh table identity.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.assets_raw")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.assets_raw")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {BRONZE}.assets_raw (up to 5 records):")
spark.table(f"{BRONZE}.assets_raw").show(5, truncate=False)


### purchasing_raw


In [ ]:
# Read the CSV as strings; keep the supplied values without business transformations.
source_file = "source-data/purchasing/purchase_orders.csv"
if config["source_layout"] == "flat":
    source_file = source_file.replace("/", "_")
raw = spark.read.csv(f"{SOURCE_ROOT}/{source_file}", header=True, inferSchema=False, mode="FAILFAST").fillna("")
# Recreate only this Bronze table so reruns start with a fresh table identity.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.purchasing_raw")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.purchasing_raw")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {BRONZE}.purchasing_raw (up to 5 records):")
spark.table(f"{BRONZE}.purchasing_raw").show(5, truncate=False)


### schedules_raw


In [ ]:
# Read the CSV as strings; keep the supplied values without business transformations.
source_file = "source-data/schedules/project_milestones.csv"
if config["source_layout"] == "flat":
    source_file = source_file.replace("/", "_")
raw = spark.read.csv(f"{SOURCE_ROOT}/{source_file}", header=True, inferSchema=False, mode="FAILFAST").fillna("")
# Recreate only this Bronze table so reruns start with a fresh table identity.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.schedules_raw")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.schedules_raw")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {BRONZE}.schedules_raw (up to 5 records):")
spark.table(f"{BRONZE}.schedules_raw").show(5, truncate=False)


### inspections_raw


In [ ]:
# Read the CSV as strings; keep the supplied values without business transformations.
source_file = "source-data/inspections/inspection_findings.csv"
if config["source_layout"] == "flat":
    source_file = source_file.replace("/", "_")
raw = spark.read.csv(f"{SOURCE_ROOT}/{source_file}", header=True, inferSchema=False, mode="FAILFAST").fillna("")
# Recreate only this Bronze table so reruns start with a fresh table identity.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.inspections_raw")
raw.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.inspections_raw")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {BRONZE}.inspections_raw (up to 5 records):")
spark.table(f"{BRONZE}.inspections_raw").show(5, truncate=False)


### Review the Bronze tables and row counts

Now we can see all the tables in the Bronze schema and the number of rows in each table. Run the next cell to check that the expected tables were created and contain data after loading the CSV files. If a table is missing or its count is unexpected, review the earlier cells and their outputs.

This is a quick loading check, not a full data-quality validation. The cell only reads the tables; it does not change any data.


In [ ]:
# Print each saved Bronze table and its current record count (read-only).
print(f"{'Bronze table':<80} {'Record count':>12}")
for table_name in sorted(table.name for table in spark.catalog.listTables(BRONZE) if not table.isTemporary):
    full_name = f"{BRONZE}.{table_name}"
    record_count = spark.table(full_name).count()
    print(f"{full_name:<80} {record_count:>12,}")
